In [2]:
from pynq import Overlay
from pynq.lib import AxiGPIO
import time

# 1. 加载硬件
overlay = Overlay("traffic_light.bit") # 换成你实际的文件名
print("✅ 硬件加载成功！")

# 2. 获取 GPIO
led = AxiGPIO(overlay.ip_dict['rgb_gpio']).channel1
led.setdirection("out")

btn = AxiGPIO(overlay.ip_dict['btn_gpio']).channel1
btn.setdirection("in")

print("🚦 交通灯启动！(绿灯20s / 黄灯3s / 红灯20s)")
print("👉 BTN0：切换紧急红灯")
print("👉 BTN1：切换夜间黄闪")

is_night_mode = False
is_emergency = False
last_btn0 = 0
last_btn1 = 0

try:
    while True:
        # 实时读取按键
        btn_val = btn.read() & 0x03
        btn0_pressed = (btn_val & 0x01)
        btn1_pressed = (btn_val & 0x02)

        # 1. 紧急模式切换 (BTN0)
        if btn0_pressed and not last_btn0:
            is_emergency = not is_emergency
            if is_emergency:
                print("🚨 进入紧急状态：锁定红灯！")
                is_night_mode = False # 互斥
            else:
                print("✅ 紧急状态解除，恢复正常。")
        last_btn0 = btn0_pressed

        # 2. 夜间模式切换 (BTN1)
        if btn1_pressed and not last_btn1:
            is_night_mode = not is_night_mode
            if is_night_mode:
                print("🌙 进入夜间黄闪")
                is_emergency = False # 互斥
            else:
                print("☀️ 恢复正常模式")
        last_btn1 = btn1_pressed

        # --- 状态执行逻辑 ---

        # 紧急模式 (红灯常亮)
        if is_emergency:
            led.write(0x01, 0x07) 
            time.sleep(0.1)
            continue

        # 夜间模式 (黄灯闪烁)
        if is_night_mode:
            led.write(0x03, 0x07) # 黄灯亮
            # 把 0.5 秒拆成 5 个 0.1 秒，随时准备检测按键退出
            for _ in range(5):
                if (btn.read() & 0x03) != 0: break
                time.sleep(0.1)
                
            led.write(0x00, 0x07) # 熄灭
            for _ in range(5):
                if (btn.read() & 0x03) != 0: break
                time.sleep(0.1)
            continue

        # 🟢 绿灯通行 (20秒)
        print("🟢 绿灯通行 (20秒)")
        for _ in range(200):
            if (btn.read() & 0x03) != 0: break
            led.write(0x02, 0x07) 
            time.sleep(0.1)
        
        if (btn.read() & 0x03) != 0: continue

        # 🟡 黄灯警告 (3秒)
        print("🟡 黄灯警告 (3秒)")
        for _ in range(30):
            if (btn.read() & 0x03) != 0: break
            led.write(0x03, 0x07) 
            time.sleep(0.1)

        if (btn.read() & 0x03) != 0: continue
            
        # 🔴 红灯禁行 (20秒)
        print("🔴 红灯禁行 (20秒)")
        for _ in range(200):
            if (btn.read() & 0x03) != 0: break
            led.write(0x01, 0x07) 
            time.sleep(0.1)

except KeyboardInterrupt:
    print("\n程序结束，安全关灯。")
    led.write(0x00, 0x07)

✅ 硬件加载成功！
🚦 交通灯启动！(绿灯20s / 黄灯3s / 红灯20s)
👉 BTN0：切换紧急红灯
👉 BTN1：切换夜间黄闪
🟢 绿灯通行 (20秒)
🌙 进入夜间黄闪
☀️ 恢复正常模式
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 

🚨 进入紧急状态：锁定红灯！
✅ 紧急状态解除，恢复正常。
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (20秒)
🟢 绿灯通行 (